# 🧠 Huấn luyện Mô hình MLP (Multi-Layer Perceptron)
**Đề tài:** Nghiên cứu ứng dụng học máy trong giám sát và phát hiện bất thường trên hệ thống mạng doanh nghiệp  
**Dữ liệu:** `data/processed/unified.parquet` (62 đặc trưng, 4 lớp: BENIGN, BruteForce, DoS/DDoS, PortScan)  
**Pipeline:** StandardScaler + MLPClassifier (Early stopping, Adam optimizer)

In [ ]:
import os
import sys
import time
import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline

# Đảm bảo import được module từ ml_engine
if '..' not in sys.path:
    sys.path.insert(0, '..')

from ml_engine.data_loader import get_train_val_test_split, CLASS_NAMES
from ml_engine.eval_utils import evaluate_predictions, measure_inference_latency, format_markdown_report

## 1. Nạp dữ liệu và Phân chia Train / Val / Test (Group-Aware)

In [ ]:
parquet_path = '../data/processed/unified.parquet'
if not os.path.exists(parquet_path):
    parquet_path = '/kaggle/input/khoaluan-nids/unified.parquet'

X_train, y_train, X_val, y_val, X_test, y_test, feature_cols = get_train_val_test_split(
    parquet_path=parquet_path,
    sample_ratio=None,
    random_state=42
)
print(f"Train size: {X_train.shape[0]:,} | Val size: {X_val.shape[0]:,} | Test size: {X_test.shape[0]:,}")
print(f"Số đặc trưng: {len(feature_cols)}")

## 2. Xây dựng Pipeline Chuẩn hóa & Khởi tạo Mạng MLP

In [ ]:
scaler = StandardScaler()
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    alpha=1e-4,
    batch_size=1024,
    learning_rate_init=0.001,
    learning_rate='adaptive',
    max_iter=30,
    early_stopping=True,
    n_iter_no_change=10,
    random_state=42,
    verbose=True
)

pipeline = Pipeline([
    ('scaler', scaler),
    ('mlp', mlp)
])

## 3. Huấn luyện mô hình

In [ ]:
start_time = time.time()
pipeline.fit(X_train, y_train)
train_duration = time.time() - start_time
print(f"Huấn luyện xong trong {train_duration:.2f} giây!")

## 4. Đánh giá toàn diện trên tập Test độc lập

In [ ]:
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)

results = evaluate_predictions(
    y_true=y_test,
    y_pred=y_pred,
    y_prob=y_prob,
    model_name="MLP(128, 64)",
    execution_time_sec=train_duration
)
latency = measure_inference_latency(pipeline, X_test)

report_md = format_markdown_report(results, latency)
print(report_md)

## 5. Vẽ đồ thị hàm mất mát (Loss Curve)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(mlp.loss_curve_, marker='o', color='purple', label='Training Loss')
plt.title('Đường cong hàm mất mát MLP (Loss Curve)')
plt.xlabel('Epochs')
plt.ylabel('Loss (Cross-Entropy)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## 6. Đóng gói Pipeline duy nhất (.joblib)

In [ ]:
save_dir = '../ml_engine/saved_models'
os.makedirs(save_dir, exist_ok=True)
joblib.dump({
    'pipeline': pipeline,
    'feature_cols': feature_cols,
    'class_names': CLASS_NAMES,
    'metrics': results
}, os.path.join(save_dir, 'mlp_pipeline.joblib'))
print("Đã đóng gói pipeline MLP thành công!")